In [2]:
import numpy as np
import pandas as pd
import os
from scipy.interpolate import interp1d

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

import tensorflow as tf
from tensorflow.keras import layers, Model

In [3]:
ESP32_DATA_PATH = "/Users/aleynagulkazdal/Desktop/esp32_data"

TARGET_FRAMES = 256
TARGET_SC = 90

WINDOW_SIZE = 128
STRIDE = 32

In [4]:
#Sadece still
USERS = [
    "aleyna",
    "damla",
    "deniz",
    "derya"
]

USER_MAP = {
    user:i
    for i, user in enumerate(USERS)
}

print(USER_MAP)

{'aleyna': 0, 'damla': 1, 'deniz': 2, 'derya': 3}


In [5]:
def parse_esp32_csv(filepath):

    rows = []

    with open(filepath, 'r') as f:
        lines = f.readlines()

    header = lines[0].strip().split(',')
    expected_cols = len(header)

    for line in lines[1:]:

        vals = line.strip().split(',')

        if len(vals) == expected_cols:
            try:
                rows.append([float(v) for v in vals])
            except:
                continue

    if len(rows) == 0:
        raise ValueError("Hiç veri okunamadı")

    raw = np.array(rows)

    raw = raw[:, 1:]

    raw = raw[:, 4:]

    real = raw[:, 0::2]
    imag = raw[:, 1::2]

    min_sc = min(real.shape[1], imag.shape[1])

    real = real[:, :min_sc]
    imag = imag[:, :min_sc]

    amp = np.sqrt(real**2 + imag**2)

    x_old = np.linspace(0, 1, amp.shape[1])
    x_new = np.linspace(0, 1, TARGET_SC)

    interpolated = np.zeros((amp.shape[0], TARGET_SC))

    for i in range(amp.shape[0]):
        f = interp1d(x_old, amp[i], kind='linear')
        interpolated[i] = f(x_new)

    return interpolated

In [6]:
def fix_length(data, target_len=TARGET_FRAMES):

    current_len = data.shape[0]

    if current_len == target_len:
        return data

    x_old = np.linspace(0, 1, current_len)
    x_new = np.linspace(0, 1, target_len)

    f = interp1d(
        x_old,
        data,
        axis=0,
        kind='linear',
        fill_value="extrapolate"
    )

    return f(x_new)

In [7]:
def create_windows(data, window_size=WINDOW_SIZE, stride=STRIDE):

    windows = []

    for start in range(
        0,
        len(data) - window_size + 1,
        stride
    ):

        end = start + window_size

        windows.append(data[start:end])

    return np.array(windows)

In [8]:
all_files = []

for user, label in USER_MAP.items():

    still_path = os.path.join(
        ESP32_DATA_PATH,
        user,
        "still"
    )

    csv_files = sorted([
        f for f in os.listdir(still_path)
        if f.endswith(".csv")
    ])

    for csv_file in csv_files:

        filepath = os.path.join(
            still_path,
            csv_file
        )

        all_files.append(
            (filepath, label)
        )

print("Toplam CSV:", len(all_files))

Toplam CSV: 80


In [9]:
paths = [x[0] for x in all_files]
labels = [x[1] for x in all_files]

train_paths, test_paths, train_labels, test_labels = train_test_split(
    paths,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Train CSV:", len(train_paths))
print("Test CSV:", len(test_paths))

Train CSV: 64
Test CSV: 16


In [10]:
X_train = []
y_train = []

for filepath, label in zip(train_paths, train_labels):

    try:

        data = parse_esp32_csv(filepath)

        data = fix_length(data)

        windows = create_windows(data)

        for w in windows:

            X_train.append(w)
            y_train.append(label)

    except Exception as e:

        print("HATA:", filepath)
        print(e)

X_train = np.array(X_train)
y_train = np.array(y_train)

print(X_train.shape)

(320, 128, 90)


In [11]:
X_test = []
y_test = []

for filepath, label in zip(test_paths, test_labels):

    try:

        data = parse_esp32_csv(filepath)

        data = fix_length(data)

        windows = create_windows(data)

        for w in windows:

            X_test.append(w)
            y_test.append(label)

    except Exception as e:

        print("HATA:", filepath)
        print(e)

X_test = np.array(X_test)
y_test = np.array(y_test)

print(X_test.shape)

(80, 128, 90)


In [12]:
X_train_norm = np.zeros_like(X_train)

for i in range(len(X_train)):

    x_min = X_train[i].min()
    x_max = X_train[i].max()

    X_train_norm[i] = (
        X_train[i] - x_min
    ) / (
        x_max - x_min + 1e-8
    )

print("Train normalize tamam")

Train normalize tamam


In [13]:
X_test_norm = np.zeros_like(X_test)

for i in range(len(X_test)):

    x_min = X_test[i].min()
    x_max = X_test[i].max()

    X_test_norm[i] = (
        X_test[i] - x_min
    ) / (
        x_max - x_min + 1e-8
    )

print("Test normalize tamam")

Test normalize tamam


In [14]:
print("\nTRAIN CSV'LER:")
for p in train_paths[:5]:
    print(os.path.basename(p))

print("\nTEST CSV'LER:")
for p in test_paths[:5]:
    print(os.path.basename(p))


TRAIN CSV'LER:
aleyna_still_02.csv
damla_still_16.csv
damla_still_06.csv
damla_still_13.csv
deniz_still_16.csv

TEST CSV'LER:
derya_still_17.csv
derya_still_14.csv
aleyna_still_11.csv
damla_still_12.csv
aleyna_still_07.csv


In [15]:
inputs = layers.Input(shape=(128, 90))

x = layers.Conv1D(
    32,
    kernel_size=5,
    activation='relu',
    padding='same'
)(inputs)

x = layers.BatchNormalization()(x)

x = layers.MaxPooling1D(2)(x)

x = layers.Dropout(0.3)(x)

x = layers.Conv1D(
    64,
    kernel_size=3,
    activation='relu',
    padding='same'
)(x)

x = layers.BatchNormalization()(x)

x = layers.MaxPooling1D(2)(x)

x = layers.Dropout(0.3)(x)

x = layers.GlobalAveragePooling1D()(x)

embedding = layers.Dense(
    64,
    activation='relu',
    name='embedding'
)(x)

outputs = layers.Dense(
    4,
    activation='softmax'
)(embedding)

model = Model(inputs, outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 90)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 128, 32)        │        14,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 64, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 64, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 32, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Dense)               │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,444 (99.39 KB)

 Trainable params: 25,252 (98.64 KB)

 Non-trainable params: 192 (768.00 B)

In [16]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [17]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

In [19]:
history = model.fit(
    X_train_norm,
    y_train,
    validation_data=(
        X_test_norm,
        y_test
    ),
    epochs=50,
    batch_size=16,
    callbacks=[early_stop]
)

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0183 - val_accuracy: 0.9375 - val_loss: 0.9872
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9969 - loss: 0.0217 - val_accuracy: 0.9375 - val_loss: 0.9567
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0358 - val_accuracy: 0.9375 - val_loss: 0.9087
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.0097 - val_accuracy: 0.9375 - val_loss: 0.7899
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9906 - loss: 0.0245 - val_accuracy: 0.7500 - val_loss: 0.7279


In [20]:
embedding_model = Model(
    model.input,
    model.get_layer("embedding").output
)

In [21]:
test_embeddings = embedding_model.predict(
    X_test_norm
)

print(test_embeddings.shape)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
(80, 64)


In [22]:
gallery = {}

for user_id in np.unique(y_train):

    user_embeddings = embedding_model.predict(
        X_train_norm[y_train == user_id]
    )

    mean_embedding = np.mean(
        user_embeddings,
        axis=0
    )

    gallery[user_id] = mean_embedding


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [23]:
ID_TO_USER = {
    v:k
    for k,v in USER_MAP.items()
}

print(ID_TO_USER)

{0: 'aleyna', 1: 'damla', 2: 'deniz', 3: 'derya'}


In [33]:
sample_idx = 0

sample = X_test_norm[sample_idx:sample_idx+1]

true_user = y_test[sample_idx]

sample_embedding = embedding_model.predict(sample)[0]

best_user = None
best_score = -1

for user_id, gallery_embedding in gallery.items():

    score = cosine_similarity(
        [sample_embedding],
        [gallery_embedding]
    )[0][0]

    print(
        ID_TO_USER[user_id],
        "->",
        round(score, 4)
    )

    if score > best_score:

        best_score = score
        best_user = user_id

print("\nGERÇEK:",
      ID_TO_USER[true_user])

print("TAHMİN:",
      ID_TO_USER[best_user])

print("SKOR:",
      round(best_score,4))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
aleyna -> 0.2241
damla -> 0.4179
deniz -> 0.3181
derya -> 0.9892

GERÇEK: derya
TAHMİN: derya
SKOR: 0.9892


In [25]:
correct = 0

all_scores = []

for i in range(len(X_test_norm)):

    sample = X_test_norm[i:i+1]

    true_user = y_test[i]

    sample_embedding = embedding_model.predict(
        sample,
        verbose=0
    )[0]

    best_user = None
    best_score = -1

    for user_id, gallery_embedding in gallery.items():

        score = cosine_similarity(
            [sample_embedding],
            [gallery_embedding]
        )[0][0]

        if score > best_score:

            best_score = score
            best_user = user_id

    all_scores.append(best_score)

    if best_user == true_user:
        correct += 1

accuracy = correct / len(X_test_norm)

print("Verification Accuracy:",
      round(accuracy * 100,2),
      "%")

Verification Accuracy: 93.75 %


In [34]:
print(
    "Min score:",
    round(min(all_scores),4)
)

print(
    "Max score:",
    round(max(all_scores),4)
)

print(
    "Mean score:",
    round(np.mean(all_scores),4)
)

Min score: 0.759
Max score: 0.997
Mean score: 0.9564


In [35]:
THRESHOLD = 0.90

correct_known = 0
rejected = 0

for i in range(len(X_test_norm)):

    sample = X_test_norm[i:i+1]

    true_user = y_test[i]

    sample_embedding = embedding_model.predict(
        sample,
        verbose=0
    )[0]

    best_user = None
    best_score = -1

    for user_id, gallery_embedding in gallery.items():

        score = cosine_similarity(
            [sample_embedding],
            [gallery_embedding]
        )[0][0]

        if score > best_score:

            best_score = score
            best_user = user_id

    if best_score < THRESHOLD:

        rejected += 1

    else:

        if best_user == true_user:

            correct_known += 1

known_accuracy = (
    correct_known /
    (len(X_test_norm) - rejected + 1e-8)
)

print("Threshold:", THRESHOLD)

print(
    "Accepted samples:",
    len(X_test_norm) - rejected
)

print(
    "Rejected samples:",
    rejected
)

print(
    "Known accuracy:",
    round(known_accuracy * 100,2),
    "%"
)

Threshold: 0.9
Accepted samples: 73
Rejected samples: 7
Known accuracy: 100.0 %


In [36]:
UNKNOWN_USER = 3   # derya

known_mask_train = y_train != UNKNOWN_USER
unknown_mask_test = y_test == UNKNOWN_USER

X_known_train = X_train_norm[known_mask_train]
y_known_train = y_train[known_mask_train]

X_unknown_test = X_test_norm[unknown_mask_test]
y_unknown_test = y_test[unknown_mask_test]

print("Known train:", X_known_train.shape)
print("Unknown test:", X_unknown_test.shape)

Known train: (240, 128, 90)
Unknown test: (20, 128, 90)


In [38]:
gallery_unknown = {}

for user_id in np.unique(y_known_train):

    user_embeddings = embedding_model.predict(
        X_known_train[y_known_train == user_id],
        verbose=0
    )

    mean_embedding = np.mean(
        user_embeddings,
        axis=0
    )

    gallery_unknown[user_id] = mean_embedding

print("Yeni gallery hazır ")

Yeni gallery hazır 


In [39]:
THRESHOLD = 0.90

rejected_unknown = 0

for i in range(len(X_unknown_test)):

    sample = X_unknown_test[i:i+1]

    sample_embedding = embedding_model.predict(
        sample,
        verbose=0
    )[0]

    best_score = -1

    for user_id, gallery_embedding in gallery_unknown.items():

        score = cosine_similarity(
            [sample_embedding],
            [gallery_embedding]
        )[0][0]

        if score > best_score:

            best_score = score

    if best_score < THRESHOLD:

        rejected_unknown += 1

unknown_rejection_rate = (
    rejected_unknown /
    len(X_unknown_test)
)

print(
    "Unknown rejection rate:",
    round(unknown_rejection_rate * 100,2),
    "%"
)

Unknown rejection rate: 100.0 %
